# 03 · Author Prediction Model
TF-IDF + behavioral features → Random Forest classifier.

In [ ]:
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
import matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/whatsapp_cleaned_data.csv")
df["datetime"] = pd.to_datetime(df["date"]+" "+df["time"], format="%m/%d/%y %I:%M %p", errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)
df["hour"]           = df["datetime"].dt.hour
df["day_of_week"]    = df["datetime"].dt.dayofweek
df["message_length"] = df["message"].str.len()
df["word_count"]     = df["message"].str.split().str.len()
df = df.dropna(subset=["message"])
print(df.shape, "unique users:", df["user"].nunique())

In [ ]:
# Keep users with >= 5 messages for reliable classification
user_counts = df["user"].value_counts()
valid_users = user_counts[user_counts >= 5].index
df = df[df["user"].isin(valid_users)].reset_index(drop=True)
print(f"Retained {df['user'].nunique()} users with >= 5 messages | {len(df)} rows")

# Label encode target
le = LabelEncoder()
y  = le.fit_transform(df["user"])
print("Classes:", len(le.classes_))

In [ ]:
# TF-IDF text features  (uni + bigrams, top 5000 features)
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000,
                        sublinear_tf=True, min_df=2)
text_features = tfidf.fit_transform(df["message"].astype(str))

# Behavioral / temporal features
num_features = csr_matrix(df[["hour","day_of_week","message_length","word_count"]].values)

# Combine
X = hstack([text_features, num_features])
print("Feature matrix:", X.shape)

In [ ]:
# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Train Random Forest
model = RandomForestClassifier(n_estimators=300, max_depth=None,
                               min_samples_leaf=1, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred)*100, 2), "%")
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy", n_jobs=-1)
print(f"5-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

In [ ]:
# Top 20 feature importances
importances = model.feature_importances_[:20]
plt.figure(figsize=(10,4))
plt.plot(importances, marker="o", color="steelblue")
plt.fill_between(range(20), importances, alpha=.3, color="steelblue")
plt.title("Top 20 Feature Importances (TF-IDF index)")
plt.xlabel("Feature Index"); plt.ylabel("Importance")
plt.tight_layout(); plt.show()

In [ ]:
# Save artifacts
joblib.dump(model,    "../models/model.pkl")
joblib.dump(tfidf,    "../models/vectorizer.pkl")
joblib.dump(le,       "../models/label_encoder.pkl")
print("Saved: model.pkl  vectorizer.pkl  label_encoder.pkl → models/")